# 15-Advanced GANs & Pitfalls

For this lesson, we will use `skimage.data.coins()` (a high-contrast grayscale image containing multiple coins of different sizes and textures). In the previous lesson, we built a standard Vanilla GAN using flat Linear layers. While mathematically elegant, if you try to deploy a Vanilla GAN in a production environment, you will immediately hit a brick wall. They are notoriously unstable, mathematically fragile, and prone to catastrophic failure.

In this lesson, we will replace the Linear layers with Deep Convolutions (DCGAN), dissect the mathematical traps of adversarial training, and rewrite the underlying calculus of the Loss Function using Wasserstein mathematics to achieve enterprise stability.

Let's set up our PyTorch environment to tame the adversarial chaos.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from skimage import data
from skimage.transform import resize
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ PyTorch Advanced GAN Environment Ready.")

✅ PyTorch Advanced GAN Environment Ready.


# 1. DCGAN (Deep Convolutional GANs)

The original 2014 GAN paper used standard Multi-Layer Perceptrons. If we want to generate high-resolution spatial images, we must introduce the geometry of Convolutions.

However, simply swapping `nn.Linear` for `nn.Conv2d` causes the Minimax game to instantly collapse. In 2015, researchers introduced the **DCGAN**, establishing the strict architectural laws required to make Convolutional GANs stable.

**The DCGAN Architectural Laws:**

1. **No Pooling Layers**: Max Pooling destroys spatial information and gradients. In the Discriminator, use strided convolutions (Stride = 2) to downsample. In the Generator, use Transpose Convolutions to upsample.
2. **Batch Normalization**: Apply BatchNorm to almost every layer in both networks to keep the extreme mathematical gradients from exploding during the Minimax war.
3. **Strict Activations**:
* Generator: Use `ReLU` everywhere, but use `Tanh` at the final output to mathematically bind the generated pixels strictly between $-1.0$ and $1.0$.
* Discriminator: Use `LeakyReLU` (e.g., negative slope of 0.2) everywhere to prevent dead gradients.

# 2. The Twin Pitfalls of Adversarial Training

Even if you build a perfect DCGAN, training it relies on a delicate balance. In practice, you will encounter two catastrophic mathematical traps.

### Pitfall 1: Vanishing Gradients (The Perfect Detective)

The Discriminator learns much faster than the Generator. It is mathematically easier to tell if an image is fake than it is to actually paint a masterpiece.
If the Discriminator becomes perfectly accurate, its output probability drops to exactly $0.0$ for fakes. When the Generator asks, *"How can I improve?"*, the mathematical derivative (the gradient) is strictly zero. The Generator goes completely blind, receives no feedback, and stops learning permanently.

### Pitfall 2: Mode Collapse (The One-Hit Wonder)

Imagine a GAN trying to learn our dataset of coins. The Generator realizes that drawing one specific, perfect silver quarter consistently fools the Discriminator. Because the Generator's only objective is "fool the Discriminator," it decides to stop trying to draw pennies, nickels, or overlapping coins. It just draws the exact same silver quarter over and over again.

The Discriminator eventually catches on and penalizes the quarter. So the Generator simply shifts to drawing a perfect penny. It chases its own tail endlessly, never learning the true diversity (the full distribution) of the dataset.

# 3. The Mathematics of WGAN (Wasserstein GAN)

In 2017, researchers discovered that these pitfalls were not an architectural flaw; they were a flaw in the fundamental calculus of the Loss Function.

Standard GANs use Binary Cross Entropy (which is based on **Jensen-Shannon (JS) Divergence**). JS Divergence measures how different two probability distributions are.

* **The Fatal Flaw**: If the Fake distribution and the Real distribution do not physically overlap in the high-dimensional space, the JS Divergence calculates the distance as a harsh, flat constant. Because a constant has a derivative of exactly zero, the gradients vanish!

### The Earth Mover's Distance

Researchers proposed throwing away JS Divergence and replacing it with the **Wasserstein Distance** (also called Earth Mover's Distance).
Imagine the Real distribution as a pile of dirt, and the Fake distribution as an empty hole. The Wasserstein distance is the minimum *cost* (mass $\times$ distance) required to physically shovel the fake dirt into the real hole.

Unlike JS Divergence, the Earth Mover's Distance provides a smooth, continuous, meaningful gradient **everywhere**, no matter how far apart the distributions are! The Generator never goes blind.

# 4. From Discriminator to Critic (1-Lipschitz)

To implement Wasserstein mathematics, we must radically alter the Discriminator. In fact, we no longer call it a Discriminator; we call it a **Critic**.

1. **No More Sigmoid**: A Discriminator outputs a probability between $0$ and $1$. A Critic outputs a raw, unbounded real number (a "score"). Higher is more real, lower is more fake.
2. **The WGAN Loss Function**: The complex logarithms of BCE are gone. The Loss is incredibly simple:

$$L_{Critic} = \mathbb{E}[C(Fake)] - \mathbb{E}[C(Real)]$$


$$L_{Generator} = -\mathbb{E}[C(Fake)]$$


3. **Weight Clipping (Lipschitz Constraint)**: To make the underlying mathematics valid, the Critic *must* be constrained so that its gradients never exceed $1.0$ (a property called 1-Lipschitz continuous). In the original WGAN paper, they forced this by brutally clipping the Critic's weights between $[-0.01, 0.01]$ after every single batch. *(Engineering Note: We also swap the Adam optimizer for RMSprop, as Adam's momentum interacts poorly with weight clipping).*

# 5. Implementing a WGAN in PyTorch

Let's build the WGAN architecture. We will define a Convolutional Critic (no Sigmoid!), a Transpose Convolutional Generator, and write the custom Wasserstein training loop that actively prevents Mode Collapse.

In [2]:
# 1. Fetch and Preprocess the Target Real Data
numpy_coins = data.coins() # Grayscale image of coins
# Resize to 64x64 to perfectly match the DCGAN Transpose architecture dimensions
resized_coins = resize(numpy_coins, (64, 64), anti_aliasing=True)

# Normalization: Generators use Tanh(), which bounds pixels between [-1.0, 1.0].
normalized_coins = (resized_coins * 2.0) - 1.0
# [Batch=1, Channels=1, Height=64, Width=64]
real_image_tensor = torch.tensor(normalized_coins, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

Z_DIM = 100
FEATURES_C = 64
FEATURES_G = 64

# 2. Define the WGAN Critic (Replacing the Discriminator)
class WGANCritic(nn.Module):
    def __init__(self, channels_img=1, features_d=64):
        super().__init__()
        self.critic = nn.Sequential(
            # Input: [Batch, 1, 64, 64] -> Output: [Batch, 64, 32, 32]
            nn.Conv2d(channels_img, features_d, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            
            # Input: [Batch, 64, 32, 32] -> Output: [Batch, 128, 16, 16]
            nn.Conv2d(features_d, features_d * 2, kernel_size=4, stride=2, padding=1),
            # WGAN prefers InstanceNorm over BatchNorm to maintain the Lipschitz constraint
            nn.InstanceNorm2d(features_d * 2), 
            nn.LeakyReLU(0.2),
            
            # Input: [Batch, 128, 16, 16] -> Output: [Batch, 256, 8, 8]
            nn.Conv2d(features_d * 2, features_d * 4, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(features_d * 4),
            nn.LeakyReLU(0.2),
            
            # Final Output: A single raw number per image (No Sigmoid!)
            # [Batch, 256, 8, 8] -> [Batch, 1, 1, 1]
            nn.Conv2d(features_d * 4, 1, kernel_size=8, stride=1, padding=0)
        )

    def forward(self, x):
        return self.critic(x).view(-1) # Flatten out the tensor to a 1D vector of scores

# 3. Define the DCGAN-style Generator
class WGANGenerator(nn.Module):
    def __init__(self, z_dim=100, channels_img=1, features_g=64):
        super().__init__()
        self.gen = nn.Sequential(
            # Input: Latent Z. Output: [Batch, 256, 8, 8]
            nn.ConvTranspose2d(z_dim, features_g * 4, kernel_size=8, stride=1, padding=0),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(),
            
            # Upsample: [Batch, 256, 8, 8] -> [Batch, 128, 16, 16]
            nn.ConvTranspose2d(features_g * 4, features_g * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(),
            
            # Upsample: [Batch, 128, 16, 16] -> [Batch, 64, 32, 32]
            nn.ConvTranspose2d(features_g * 2, features_g, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(features_g),
            nn.ReLU(),
            
            # Final Layer: Output image [Batch, 1, 64, 64]
            nn.ConvTranspose2d(features_g, channels_img, kernel_size=4, stride=2, padding=1),
            nn.Tanh() # Mathematically bounds pixels to [-1, 1]
        )

    def forward(self, x):
        return self.gen(x)

# 4. The WGAN Training Loop Architecture
print("--- ⚙️ Initializing WGAN Mathematics ---")
CRITIC_ITERATIONS = 5 # The Critic MUST train 5 times for every 1 Generator step!
WEIGHT_CLIP = 0.01
BATCH_SIZE = 4

critic = WGANCritic(channels_img=1, features_d=FEATURES_C)
gen = WGANGenerator(z_dim=Z_DIM, channels_img=1, features_g=FEATURES_G)

# RMSprop is mathematically required for standard WGAN (Adam's momentum breaks the clipping)
opt_critic = optim.RMSprop(critic.parameters(), lr=5e-5) 
opt_gen = optim.RMSprop(gen.parameters(), lr=5e-5)

print("✅ Critic and Generator instantiated. Notice: No BCELoss Function exists!")

# 5. Simulating a single WGAN Training Step
real_batch = real_image_tensor.repeat(BATCH_SIZE, 1, 1, 1) 
noise = torch.randn(BATCH_SIZE, Z_DIM, 1, 1)

print("\n--- ⚖️ Phase 1: Training the Critic (5 Iterations) ---")
for _ in range(CRITIC_ITERATIONS):
    fake_images = gen(noise)
    
    # Critic evaluates Real and Fake
    critic_real = critic(real_batch)
    critic_fake = critic(fake_images.detach()) 
    
    # WGAN Critic Loss Equation: E[Critic(Fake)] - E[Critic(Real)]
    # We want to MAXIMIZE the difference, so we MINIMIZE the negative difference
    loss_critic = -(torch.mean(critic_real) - torch.mean(critic_fake))
    
    opt_critic.zero_grad()
    loss_critic.backward()
    opt_critic.step()
    
    # The Lipschitz Hack: Brutal Weight Clipping
    for p in critic.parameters():
        p.data.clamp_(-WEIGHT_CLIP, WEIGHT_CLIP)

print(f"Critic Loss (Wasserstein Distance): {loss_critic.item():.4f}")
print("Insight: The weights are clipped to [-0.01, 0.01]. The Earth Mover's gradient is strictly stable.")

print("\n--- 🎨 Phase 2: Training the Generator (1 Iteration) ---")
# 1. Generator creates new images
fake_images_for_gen = gen(noise)

# 2. Critic evaluates them
output = critic(fake_images_for_gen)

# 3. WGAN Generator Loss Equation: -E[Critic(Fake)]
# We want the Critic to output a massive positive score for our fakes!
loss_gen = -torch.mean(output)

opt_gen.zero_grad()
loss_gen.backward()
opt_gen.step()

print(f"Generator Loss: {loss_gen.item():.4f}")
print("Success! The Generator received a pure, meaningful gradient despite the Critic being highly trained.")

--- ⚙️ Initializing WGAN Mathematics ---
✅ Critic and Generator instantiated. Notice: No BCELoss Function exists!

--- ⚖️ Phase 1: Training the Critic (5 Iterations) ---
Critic Loss (Wasserstein Distance): -39.3696
Insight: The weights are clipped to [-0.01, 0.01]. The Earth Mover's gradient is strictly stable.

--- 🎨 Phase 2: Training the Generator (1 Iteration) ---
Generator Loss: 18.6888
Success! The Generator received a pure, meaningful gradient despite the Critic being highly trained.


## Real-World Use Case or Analogy:

Think of the transition from Vanilla GAN to WGAN like **Grading an Art Student**:

* **Vanilla GAN (The Harsh Pass/Fail Judge)**: The Discriminator looks at the student's painting of a coin. If it isn't an absolute perfect replica, the Discriminator stamps `0.0 (FAIL)`. The student asks, *"What did I do wrong? Was the color off? Was the perspective bad?"* The Judge refuses to explain and just repeats `FAIL`. The student learns nothing, goes blind, and traces a single quarter forever just to get a pass (Mode Collapse / Vanishing Gradients).
* **Wasserstein GAN (The Constructive Art Critic)**: The Critic doesn't use Pass/Fail. It gives the painting a continuous, unbounded score, like `-452`. The student asks how to improve. The Critic gives exact, continuous geometric feedback: *"Your score will improve to `-400` if you move this specific shadow 2 inches to the left (Earth Mover's Distance)."* Even if the painting is terrible static noise, the Critic can always tell the student exactly which direction to move to get mathematically closer to reality. The student steadily explores the entire artistic space without ever failing catastrophically.